In [1]:
# Import packages
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options

from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.common.exceptions import ElementNotInteractableException

from bs4 import BeautifulSoup

import time, re, requests
import pandas as pd

In [2]:
# Open the browser without showing it on screen
opts = Options()
opts.add_argument("--headless=new")
opts.add_argument("window-size=2560,1600")

# Open the Chrome browser
driver = webdriver.Chrome() #(options=opts)

# Navigate to the Project Gutenberg homepage
url = "https://www.gutenberg.org/"
driver.get(url)

# Set the implicit waiting time to 0.5 seconds
driver.implicitly_wait(0.5)
# Set the explicit wwaiting time to 20 seconds
wait = WebDriverWait(driver, 20)

In [3]:
# Find the navigation bar links
nav_bar = driver.find_elements(By.CSS_SELECTOR, ".main-links a")
if nav_bar:
    for link in nav_bar:
        # Print the navigation bar links
        print(link.text, link.get_attribute("href"))

 https://www.gutenberg.org/browse/scores/top
 https://www.gutenberg.org/ebooks/categories
Reading Lists https://www.gutenberg.org/ebooks/bookshelf/
Search Options https://www.gutenberg.org/ebooks/


In [4]:
# Navigate to the Reading Lists
reading_lists = driver.find_element(By.CSS_SELECTOR, "a[class='link-reading-lists']")
reading_lists.click()

# Pause execution for 5 seconds
time.sleep(5)

In [5]:
# Navigate to the Main Categories webpage
main_categories = driver.find_element(By.CSS_SELECTOR, "div[class='page_content'] p a")
main_categories.click()

# Pause execution for 5 seconds
time.sleep(5)

In [ ]:
# Navigate to the Science-Fiction & Fantasy webpage
sf_fantasy = driver.find_element(By.XPATH, "/html/body/div[1]/div/div/div[1]/ul/li[14]/a")
sf_fantasy.click()

# Pause execution for 5 seconds
time.sleep(5)

In [ ]:
def scrape_rows():
    """
    Scrape the novels from a specific webpage.

    Returns:
        dict: Scraped novel values.
    """
    # Load Selenium's page_source into BeautifulSoup
    document = BeautifulSoup(driver.page_source, "html.parser")

    index = 2
    authors = []
    # Iterate over each novel on the webpage
    for book in document.select(".booklink"):
        try:
            # Append the author metadata to the authors list if it exists
            if driver.find_element(By.XPATH, f'//*[@id="content"]/div/ul/li[{index}]/a/span[2]/span[3]'):
                authors.append(driver.find_element(By.XPATH, f'//*[@id="content"]/div/ul/li[{index}]/a/span[2]/span[2]').text)
        except NoSuchElementException:
            # Append NaN value to the authors list if there is no author metadata
            authors.append("NaN")
        index += 1

    return {
        "titles": document.select(".title"),
        "authors": authors,
        "downloads": document.select(".extra"),
        "texts": driver.find_elements(By.CSS_SELECTOR, ".booklink a")
    }

In [ ]:
def create_text():
    """
    Turn the scraped rows into text.

    Returns:
        dict: Novel values in text form.
    """
    # Assign scrape_rows() function to scrape_rows_result variable
    scrape_rows_result = scrape_rows()

    return {
        "title_text": [title.text for title in scrape_rows_result["titles"]],
        "author_text": [author for author in scrape_rows_result["authors"]],
        "download_text": [download.text for download in scrape_rows_result["downloads"]],
        "text_text": [text.get_attribute("href") for text in scrape_rows_result["texts"]]
    }

In [ ]:
def collect_data():
    """
    Iterates over novels and scrapes its metadata.
    
    Returns:
        List: Novel metadata.
    """
    # Assign create_text() function to create_text_result variable
    create_text_result = create_text()
    # Count the amount of novels
    row_count = len(driver.find_elements(By.CSS_SELECTOR, ".booklink"))

    # Append metadata from specific webpage to list
    specific_page = []
    row_index = 0
    for row in range(row_count):
        specific_page.append({
            "Title": create_text_result["title_text"][row_index],
            "Author": create_text_result["author_text"][row_index],
            "Downloads": create_text_result["download_text"][row_index],
            "Text": create_text_result["text_text"][row_index]
        })
        row_index += 1

    return specific_page

In [ ]:
# Instantiate the df_sf_fantasy DataFrame
df_sf_fantasy = pd.DataFrame()

# The amount of webpages
page_length = 169

# Iterate through the webpages
for page in range(page_length):
    # Create the df_append DataFrame, based on the results of the scrape_page() function
    df_append = pd.DataFrame(collect_data())
    # Append the df_append DataFrame to the df_sf_fantasy DataFrame
    df_sf_fantasy = pd.concat([df_sf_fantasy, df_append], ignore_index=True)

    try:
        # Click on the next page link if it exists
        driver.find_element(By.CSS_SELECTOR, "a[title='Go to the next page of results.']").click()
        time.sleep(3)
    except NoSuchElementException:
        # Do nothing if the next page link does not exist
        pass

# Show the df_sf_fantasy DataFrame
df_sf_fantasy

,Title,Author,Downloads,Text
0,"Frankenstein; Or, The Modern Prometheus",Mary Wollstonecraft Shelley,264413 downloads,https://www.gutenberg.org/ebooks/84
1,The King in Yellow,Robert W. Chambers,68842 downloads,https://www.gutenberg.org/ebooks/8492
2,Dracula,Bram Stoker,42071 downloads,https://www.gutenberg.org/ebooks/345
3,The Wonderful Wizard of Oz,L. Frank Baum,24328 downloads,https://www.gutenberg.org/ebooks/55
4,"Frankenstein; Or, The Modern Prometheus",Mary Wollstonecraft Shelley,20274 downloads,https://www.gutenberg.org/ebooks/41445
...,...,...,...,...
4207,Tuhatvuotinen valtakunta : Nelinäytöksinen nä...,Upton Sinclair and Heikki Välisalmi,160 downloads,https://www.gutenberg.org/ebooks/76543
4208,Tape Jockey,Tom Leahy,159 downloads,https://www.gutenberg.org/ebooks/32583
4209,Ennen Aatamia (Finnish),Jack London,157 downloads,https://www.gutenberg.org/ebooks/56329
4210,Taikapeili : Nelinäytöksinen satunäytelmä (Fi...,Larin-Kyösti,146 downloads,https://www.gutenberg.org/ebooks/71912


In [109]:
# Make a copy of the df_sf_fantasy DataFrame
df_sf_fantasy_clean = df_sf_fantasy.copy()

# Replace ebooks with cache/epub in the df_sf_fantasy_clean['Text'] column
df_sf_fantasy_clean['Text'] = df_sf_fantasy_clean['Text'].replace(to_replace=r'ebooks', value='cache/epub', regex=True)
# Extract the numbers of the novels
df_sf_fantasy_clean['Number'] = df_sf_fantasy_clean['Text'].str.extract(r"(\d+)")
# Add .txt at the end of each row of the df_sf_fantasy_clean['Number'] column
df_sf_fantasy_clean['Number'] = df_sf_fantasy_clean['Number'].replace(to_replace=r'$', value='.txt', regex=True)
# Add /pg at the beginning of each row of the df_sf_fantasy_clean['Number'] column
df_sf_fantasy_clean['Number'] = df_sf_fantasy_clean['Number'].replace(to_replace=r'^', value='/pg', regex=True)
# Combine the df_sf_fantasy_clean['Text'] and df_sf_fantasy_clean['Number'] columns
df_sf_fantasy_clean['Plain Text UTF-8 Link'] = df_sf_fantasy_clean['Text'] + df_sf_fantasy_clean['Number']
# Delete 'downloads' from the df_sf_fantasy_clean['Downloads'] column
df_sf_fantasy_clean['Downloads'] = df_sf_fantasy_clean['Downloads'].replace(to_replace=r' downloads', value='', regex=True)
# Delete the df_sf_fantasy_clean['Text'] and df_sf_fantasy_clean['Number'] columns
df_sf_fantasy_cleaned = df_sf_fantasy_clean.drop(['Text', 'Number'], axis=1)

# Show the df_sf_fantasy_cleaned DataFrame
df_sf_fantasy_cleaned

,Title,Author,Downloads,Plain Text UTF-8 Link
0,"Frankenstein; Or, The Modern Prometheus",Mary Wollstonecraft Shelley,264413,https://www.gutenberg.org/cache/epub/84/pg84.txt
1,The King in Yellow,Robert W. Chambers,68842,https://www.gutenberg.org/cache/epub/8492/pg84...
2,Dracula,Bram Stoker,42071,https://www.gutenberg.org/cache/epub/345/pg345...
3,The Wonderful Wizard of Oz,L. Frank Baum,24328,https://www.gutenberg.org/cache/epub/55/pg55.txt
4,"Frankenstein; Or, The Modern Prometheus",Mary Wollstonecraft Shelley,20274,https://www.gutenberg.org/cache/epub/41445/pg4...
...,...,...,...,...
4207,Tuhatvuotinen valtakunta : Nelinäytöksinen nä...,Upton Sinclair and Heikki Välisalmi,160,https://www.gutenberg.org/cache/epub/76543/pg7...
4208,Tape Jockey,Tom Leahy,159,https://www.gutenberg.org/cache/epub/32583/pg3...
4209,Ennen Aatamia (Finnish),Jack London,157,https://www.gutenberg.org/cache/epub/56329/pg5...
4210,Taikapeili : Nelinäytöksinen satunäytelmä (Fi...,Larin-Kyösti,146,https://www.gutenberg.org/cache/epub/71912/pg7...
